<a href="https://colab.research.google.com/github/christianlocher/llmrag/blob/main/IV_ChromaDB_Basis_fuer_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Package Download

In [1]:
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 72.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Restart Enviroment!

In [1]:

# Necessary for Colab
#Package contains PDF library
!pip install -U chromadb pypdf langchain-community


import requests
from io import BytesIO
from langchain_community.document_loaders import PyPDFLoader



  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached pypdf-6.11.0-py3-none-any.whl.metadata (7.2 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.41.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.1-py3-none-any.wh

#I Text Chunking

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

DATA_PATH = r"sample_data"
loader = PyPDFDirectoryLoader(DATA_PATH)

documents_from_url = loader.load()


#I-3 SpacyTextSplitter

In [3]:
from langchain_text_splitters import SpacyTextSplitter
import spacy.cli
from langchain_core.documents import Document
from datetime import datetime

spacy.load('de_core_news_sm')

# be careful to use the right dictionary, see: https://spacy.io/models
text_splitter = SpacyTextSplitter(
    pipeline="de_core_news_sm",
    chunk_size=500,
    chunk_overlap=50
    )

rawtext = text_splitter.split_documents(documents_from_url)

chunks = []

for i, doc_chunk in enumerate(rawtext):
    # Create a new Document object with added metadata
    new_metadata = {
        "chunk_id": i,
        "source": "",
        "orig_source": "AnySource",
        "mandant": "OtherCompany",
        "last_updated": datetime.now().isoformat(),
        "language": "de",
        **doc_chunk.metadata # Include existing metadata from the chunk
    }
    doc = Document(
        page_content=doc_chunk.page_content, # Correctly access content from the current Document object
        metadata=new_metadata
    )
    chunks.append(doc)


i = 0

for chunk in chunks:
    print(chunk)
    print('####################################################################')

    i += 1

page_content='Basisnormen für Tischler und Schreiner 
Fachregeln 


Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)".

Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 


Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden.

Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich.' metadata={'chunk_id': 0, 'source': 'sample_data/13_Basisnormen-für-Tischler-und-Schreiner.pdf', 'orig_source': 'AnySource', 'mandant': 'OtherCompany', 'last_updated': '2026-05-14T17:35:38.217511', 'language': 'de', 'producer': 'Scanner System Image Conversion', 'creator': 'Scanner System', 'creationdate': '2019-08-20T11:09:22+01:00', 'moddate': '2019-08-20T11:09:22+01:00', 'total_pages': 12, 'page': 0, 'page_label': '1'}
####################################################################
page_content='Normen könne

#III-1 Create Embeddings using Qwen3.5 0.6b and Insert into Vector Database

In this example, embeddings are created using a built-in model.


In [4]:
import chromadb

# setting the environment
# By default, Chromadb uses all-MiniLM-L6-v2 do create embeddings

CHROMA_PATH = r"chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name="Furniture")

# preparing to be added in chromadb

documents = []
metadata = []
ids = []

i = 0

for chunk in chunks:
    documents.append(chunk.page_content)
    ids.append("ID"+str(i))
    metadata.append(chunk.metadata)
    print(chunk.page_content)
    print('############################')

    i += 1

# adding to chromadb


collection.upsert(
    documents=documents,
    metadatas=metadata,
    ids=ids
)

Basisnormen für Tischler und Schreiner 
Fachregeln 


Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)".

Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 


Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden.

Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich.
############################
Normen können über 
www.beuth.de  bezogen werden.

Sinnvoll ist dabei 
meist die Anschaffung von DIN-Taschenbüchern, in 
welchen Normen nach Themen zusammengestellt 
sind.

So gibt es z. B. das „Normenhandbuch Tischler-
und Schreinerhandwerk" in welchem viele Normen 
abgebildet sind.

In den DIN-Taschenbüchern 359 und 
360 finden sich z. B. Nomen für die Holzwirtschaft.
############################
Wer viel mit Normen arbeiten muss, kann sich auch 
bestimmter DIN-Portale bedienen, auf welchen man 
zah

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 26.7MiB/s]


In [10]:
while True:

  user_query = input("How can I help you?")

  if user_query.lower() == "quit":
            break

  results = collection.query(
      query_texts=[user_query],
      n_results=10

  )

  print(results['documents'])
  print(results['metadatas'])



How can I help you?Welche Normen gelten für Tischler?
[['Basisnormen für Tischler und Schreiner \nFachregeln \n\n\nNormen gelten nicht selten als „allgemein anerkannte \nRegel der Technik (aaRdT)".\n\nDaher ist es notwendig, \njene Normen zu kennen, die für die jeweiligen Auf-\nträge relevant sind. \n\n\nMit dieser kleinen Zusammenstellung soll ein grober \nÜberblick zu wichtigen Normen für Tischler und \nSchreiner gegeben werden.\n\nDa Normen dem Copy-\nright unterliegen, ist ein Abdruck von Normen an die-\nser Stelle nicht möglich.', 'Die Suchergebnisse sind auf Normen \nbeschränkt, die für den Wirtschaftsbereich Handwerk \nund für einzelne Branchen relevant sind.\n\nDas Herun-\nterladen von Normen ist kostenpflichtig. \n\n\nBeim Suchen nach Normen für den Wirtschaftsbereich \n„Tischler" werden über 170 Normen (Stand 2017)\n\nan-\ngezeigt. \n\n\nIm Folgenden führen wir einige Normen auf, die be-\nsondere Bedeutung für Tischler/\n\nSchreiner haben.', '10.2 und 10.3), \nWände, Böden, H

#IV-2 FIltered Request

In [5]:
while True:

  user_query = input("How can I help you?")

  if user_query.lower() == "quit":
            break
  results = collection.query(
      query_texts=[user_query],
      n_results=10,
      #where={"mandant": "OtherCompany"}
      where={
        "$and": [
            {"page": {"$gte": 0 }},
            {"page": {"$lte": 12 }},
            {"mandant": "OtherCompany"}
        ]
    }

  #More filtering options: https://docs.trychroma.com/docs/querying-collections/metadata-filtering
  #More on search: https://docs.trychroma.com/cloud/search-api/search-basics
  )

  print(results['documents'])
  print(results['metadatas'])


How can I help you?Welche Normen gibt es
[['c)\n\nSonstige Normen können ggf. herangezogen wer-\nden.\n\nSie sind jedoch nicht verbindlich einzuhalten.\n\nAb-\nweichungen sollten aber auch hier begründet werden. \n\n\nDa im Streitfall gerne auf Normen zurückgegriffen \nwird, ist es sinnvoll zu recherchieren, ob es für das ge-\nplante Produkt eine Norm gibt.\n\nOft wird bei Auseinan-\ndersetzungen angenommen, dass auch die rechtlich \nnicht verbindlichen Normen als „anerkannte Regel der \nTechnik" gelten.\n\nDies ist oft — aber nicht immer der \nFall.', 'b) Normen, welche von der jeweiligen Branche i. d. R. \nangewendet werden gelten als „allgemein anerkannte \nRegel der Technik".\n\nMit diesem Status ist die Anwen-\ndung verbindlich — eine Nichtbeachtung sollte abge-\nsprochen und begründet werden. \n\n\n© Wolfgang Heer 1', 'Was regelt die Norm? \n\n\nDie Norm legt die Anforderungen an einige Eigen-\nschaften fest, die für alle Typen unbeschichteter Fa-\nserplatten nach EN 316 gleich s

# IV -3 Re-Ranked Results - Example! (Adaption for compatibility with IV-1/2 necessary)



In [6]:
from google.colab import ai
import json

while True:

  user_query = input("How can I help you?")

  if user_query.lower() == "quit":
            break
  initial_results = collection.query(
      query_texts=[user_query],
      n_results=10

  #More filtering options: https://docs.trychroma.com/docs/querying-collections/metadata-filtering

  )


  initial_results = collection.query(
    query_texts=[user_query],
    n_results=5 # Fetching top 5 for the LLM to rerank
  )

  retrieved_docs = initial_results['documents'][0]
  retrieved_ids = initial_results['ids'][0]
  retrieved_metadata = initial_results['metadatas'][0]

  print("--- Initial ChromaDB Retrieval ---")
  for doc_id, doc in zip(retrieved_ids, retrieved_docs):
    print(f"{doc_id}: {doc}")


  # ---------------------------------------------------------
  # Step 3: Construct the Reranking Prompt
  # ---------------------------------------------------------
  prompt = f"""
  You are an expert search reranker. Your task is to evaluate the relevance of the following documents to the user's query and re-rank them from most relevant to least relevant.

  User Query: "{user_query}"

  Documents to evaluate:
  """

  for doc_id, doc in zip(retrieved_ids, retrieved_docs):
    prompt += f"- ID: {doc_id} | Content: {doc}\n"

  prompt += """
  Return ONLY a raw JSON array of strings representing the document IDs in the new, reranked order.
  Do not include markdown formatting, explanations, or any other text.
  Example format: ["doc3", "doc1", "doc5", "doc2", "doc4"]
  """
  # ---------------------------------------------------------
  # Step 4: Call Google Colab AI to Rerank
  # ---------------------------------------------------------
  # Call the built-in Colab AI model
  response = ai.generate_text(prompt)

  # Safely extract the text (handling both object and string return types)
  ai_output = getattr(response, 'text', str(response))

  # ---------------------------------------------------------
  # Step 5: Parse Output and Reorder
  # ---------------------------------------------------------
  try:
      # Clean the response just in case the LLM wrapped it in markdown ticks
      clean_output = ai_output.replace('```json', '').replace('```', '').strip()
      reranked_ids = json.loads(clean_output)

      # Map the IDs back to the actual documents
      doc_mapping = dict(zip(retrieved_ids, retrieved_docs))
      reranked_docs = [doc_mapping[doc_id] for doc_id in reranked_ids if doc_id in doc_mapping]

      top_5_ids = reranked_ids[:5]
      top_5_docs = reranked_docs[:5]

      print("\n--- AI Reranked Output ---")
      for doc_id, doc in zip(top_5_ids, top_5_docs):
          print(f"{doc_id}: {doc}")

  except json.JSONDecodeError:
      print("\n[Error] Failed to parse JSON. The model may have returned conversational text.")
      print("Raw AI Output:", ai_output)

How can I help you?Welche Normen gibt es
--- Initial ChromaDB Retrieval ---
ID11: c)

Sonstige Normen können ggf. herangezogen wer-
den.

Sie sind jedoch nicht verbindlich einzuhalten.

Ab-
weichungen sollten aber auch hier begründet werden. 


Da im Streitfall gerne auf Normen zurückgegriffen 
wird, ist es sinnvoll zu recherchieren, ob es für das ge-
plante Produkt eine Norm gibt.

Oft wird bei Auseinan-
dersetzungen angenommen, dass auch die rechtlich 
nicht verbindlichen Normen als „anerkannte Regel der 
Technik" gelten.

Dies ist oft — aber nicht immer der 
Fall.
ID10: b) Normen, welche von der jeweiligen Branche i. d. R. 
angewendet werden gelten als „allgemein anerkannte 
Regel der Technik".

Mit diesem Status ist die Anwen-
dung verbindlich — eine Nichtbeachtung sollte abge-
sprochen und begründet werden. 


© Wolfgang Heer 1
ID73: Was regelt die Norm? 


Die Norm legt die Anforderungen an einige Eigen-
schaften fest, die für alle Typen unbeschichteter Fa-
serplatten nach EN 316